# App0 Ollama 后端准备指南 — 教案

---

## 课程信息

| 项目 | 内容 |
|:---|:---|
| **课程名称** | App0 Ollama 后端准备指南 |
| **预计时长** | ~25 分钟 |
| **源文件** | `Applications/PREPARE_OLLAMA.ipynb` |
| **难度** | 入门级（零基础友好） |
| **性质** | 环境准备课（非编码课），为后续 App 系列提供本地 LLM 后端 |

---

## 时间表

| 时间段 | 内容 | Cell 范围 | 时长 |
|:---|:---|:---|:---|
| 00:00–05:00 | 开场 + 什么是 Ollama + 为什么需要它 | Cell 0 | 5 min |
| 05:00–12:00 | 安装 Ollama + 版本验证 | Cell 1 | 7 min |
| 12:00–18:00 | 拉取模型 qwen3:4b + 等待下载 | Cell 2 + Cell 6 | 6 min |
| 18:00–22:00 | 启动服务 + 检查服务状态 | Cell 3 + Cell 4 | 4 min |
| 22:00–25:00 | 在 notebook 中调用 + 总结 + Q&A | Cell 5 | 3 min |

---

## 课前准备清单

- [ ] 讲师电脑已安装 Ollama 并拉取 `qwen3:4b` 模型（避免课上等下载）
- [ ] 确认 `ollama --version` 可正常输出版本号
- [ ] 确认 `http://localhost:11434/api/tags` 返回 JSON（服务正在运行）
- [ ] 准备截图/录屏作为备用演示素材（防止学生网络慢无法现场下载）
- [ ] 确认教室/远程环境网络畅通（模型下载约 2.5GB）
- [ ] 备用方案：若学生无法安装 Ollama，准备 OpenAI API key 作为替代后端

## 段落 0：开场 + 什么是 Ollama + 为什么需要它（Cell 0）

---

📍 **展示 Cell 0**

⏱ **时间分配：5 分钟**

🎯 **本段目标**
- 让学生理解 Ollama 是什么、为什么选它作为本课程的 LLM 后端
- 建立学习预期：本节结束后，每个人都能在本地跑大模型
- 消除「大模型需要很贵的 GPU」的误解

🗣 **讲课话术**

> 大家好！在正式进入应用开发之前，我们要先把「引擎」装好。
>
> 后面的 App 系列——RAG、Agent、Function Calling 等等——都需要调用一个 LLM。我们有两个选择：一是用 OpenAI 的 API，二是在本地跑一个模型。
>
> 今天我们选第二条路——**Ollama**。它是什么呢？你可以把它理解成「大模型界的 Docker」。Docker 让你一行命令跑起一个数据库，Ollama 让你一行命令跑起一个大模型。
>
> **为什么选 Ollama？** 三个原因：
> 1. **免费** —— 不需要 API key，不按 token 收费，学习阶段随便跑
> 2. **简单** —— 安装就是下一个安装包，拉模型就像 `docker pull` 一样一行命令
> 3. **隐私** —— 数据全部在本地，不用担心发到第三方服务器
>
> Cell 0 说得很清楚：这个 notebook 就是为后续应用类 notebook 做准备的。大家把它当成一个「开机检查清单」就行。
>
> **互动**：有多少同学之前在本地跑过大模型的？（举手/打 1）——没关系，今天 25 分钟搞定，跟着走就行。

👀 **输出要点**
- Cell 0 是纯 Markdown，无代码输出
- 核心信息：Ollama 用于为应用类 notebook 提供 LLM 后端

❓ **预判问题**

Q: 我电脑没有独立显卡能跑吗？
A: 能！Ollama 支持 CPU 推理。我们用的 qwen3:4b 是 4B 参数的小模型，8GB 内存就够了。有 GPU 当然更快，但没有也完全能用。

Q: 和直接调 OpenAI API 比有什么区别？
A: 功能上差不多，后续代码里我们用 `get_llm_backend()` 统一封装了，切换后端只需改一个参数。区别在于：本地模型免费但质量稍低，API 收费但质量高。学习阶段建议先用 Ollama，省钱。

Q: 为什么不用 llama.cpp 或 vLLM？
A: Ollama 底层就是 llama.cpp，但它封装了一层 HTTP API 服务，更方便 notebook 调用。vLLM 更适合生产环境的大规模部署，学习阶段用 Ollama 就够了。

➡️ **转场**

> 好，概念清楚了。接下来第一步——把 Ollama 装到你的电脑上。

## 段落 1：安装 Ollama + 版本验证（Cell 1）

---

📍 **展示 Cell 1**

⏱ **时间分配：7 分钟**（含学生实操安装时间）

🎯 **本段目标**
- 每个学生完成 Ollama 安装
- 通过 `ollama --version` 验证安装成功
- 理解 Ollama 安装后会自动注册为系统服务

🗣 **讲课话术**

> 第一步，安装。大家打开浏览器，访问 **https://ollama.ai/download**。
>
> 页面会自动识别你的操作系统。Windows 用户下载 exe 安装包，双击运行就行；Mac 用户下载 dmg，拖到 Applications；Linux 用户一行命令 `curl -fsSL https://ollama.com/install.sh | sh`。
>
> 安装完成后，打开终端（Windows 用 PowerShell 或 CMD，Mac/Linux 用 Terminal），输入：
>
> ```
> ollama --version
> ```
>
> 看到类似 `ollama version 0.x.x` 的输出就说明安装成功了。Cell 1 里有一张截图（`assets/version.png`），大家对照一下。
>
> **类比**：安装 Ollama 就像安装 Docker Desktop——装完之后它会在后台默默运行一个服务，随时等你调用。你不需要每次手动启动它。
>
> **互动**：大家装好了吗？在终端里跑一下 `ollama --version`，把版本号发出来让我看看。——很好，都是 0.x 以上的版本就没问题。

👀 **输出要点**
- Cell 1 是 Markdown + 截图，无代码运行输出
- 截图 `assets/version.png`：展示 `ollama --version` 的终端输出
- 关键验证命令：`ollama --version`

❓ **预判问题**

Q: Windows 安装完在哪里找 ollama 命令？
A: 安装程序会自动把 ollama 加入系统 PATH。如果 PowerShell 找不到，重启一下终端（关掉重新打开）。实在不行，去 `C:\Users\你的用户名\AppData\Local\Programs\Ollama` 找。

Q: Mac 上安装后提示「无法验证开发者」怎么办？
A: 去「系统偏好设置 -> 安全性与隐私 -> 通用」，点击「仍要打开」。这是 macOS 对未签名应用的常规提示。

Q: 我之前装过旧版本，需要卸载重装吗？
A: 不需要，直接安装新版会覆盖旧版。但如果版本差异很大（比如 0.1.x 升到 0.5.x），建议先卸载再安装。

Q: 安装包有多大？
A: Windows/Mac 安装包大约 100-200MB，装完占用约 500MB。模型文件是另算的，qwen3:4b 大约 2.5GB。

➡️ **转场**

> 安装搞定了！但现在 Ollama 只是一个「空壳」，里面还没有模型。接下来我们要拉取一个模型——就像 Docker 需要 pull 镜像一样。

## 段落 2：拉取模型 qwen3:4b（Cell 2 + Cell 6）

---

📍 **展示 Cell 2 和 Cell 6**

⏱ **时间分配：6 分钟**（含下载等待时间）

🎯 **本段目标**
- 每个学生成功拉取 `qwen3:4b` 模型
- 理解模型命名规则（模型名:参数规模）
- 了解模型文件的大小和存储位置

🗣 **讲课话术**

> 现在我们来下载模型。打开终端，输入：
>
> ```
> ollama pull qwen3:4b
> ```
>
> 大家看 Cell 2，就这一行命令。`qwen3` 是阿里通义千问的第三代开源模型，`:4b` 表示 40 亿参数的版本。
>
> **类比**：这就像 `docker pull nginx:latest` 一样——冒号前面是「镜像名」，后面是「标签」。在 Ollama 的世界里，标签通常是参数规模：4b 就是 40 亿参数，8b 就是 80 亿参数。
>
> **为什么选 qwen3:4b？** 两个原因：
> 1. **够小** —— 下载只有 2.5GB 左右，运行时占用约 3-4GB 内存/显存，大多数电脑都能跑
> 2. **够用** —— 4B 参数对于学习 RAG、Agent 这些应用概念完全够了，响应速度也快
>
> Cell 6 有一张下载过程的截图（`assets/qwen3_4b.png`），大家可以对照看进度条。下载速度取决于你的网络，一般几分钟就好。
>
> **互动**：大家开始 pull 了吗？看到进度条在走就对了。下载的时候我们可以聊聊——有没有同学用过其他开源模型？Llama、Mistral、Gemma 这些？

👀 **输出要点**
- Cell 2 是 Markdown，展示拉取命令 `ollama pull qwen3:4b`
- Cell 6 展示下载过程截图 `assets/qwen3_4b.png`，包含进度条界面
- 下载完成后终端会显示模型摘要信息（文件大小、量化方式等）

❓ **预判问题**

Q: 下载太慢了怎么办？
A: 检查网络。如果在国内，可以尝试设置代理。或者课后在网络好的环境下再下载，今天先看讲师的演示。

Q: 模型下载到哪里了？占多少空间？
A: 默认路径——Windows: `C:\Users\你的用户名\.ollama\models`，Mac/Linux: `~/.ollama/models`。qwen3:4b 大约占 2.5GB 磁盘空间。

Q: 能同时下载多个模型吗？
A: 可以开多个终端窗口分别 pull，但会抢带宽。建议一个一个来。

Q: 4b 参数会不会太小，效果很差？
A: 对于课程演示够用了。我们关注的是应用架构（RAG 流水线、Agent 工具调用等），不是模型本身的智力。如果你电脑配置好，可以课后试 `qwen3:8b` 或 `qwen3:14b`。

Q: 我之前已经 pull 过了，需要重新下载吗？
A: 不需要。如果已有相同版本，`ollama pull` 会提示 already up to date，秒完成。

➡️ **转场**

> 模型下载好了。但光有模型还不行——Ollama 需要有一个「服务」在后台运行，才能接收请求。接下来我们确认一下服务是否正常。

## 段落 3：启动服务 + 检查服务状态（Cell 3 + Cell 4）

---

📍 **展示 Cell 3 和 Cell 4**

⏱ **时间分配：4 分钟**

🎯 **本段目标**
- 确认每个学生的 Ollama 服务正在运行
- 掌握两种验证服务状态的方法：`ollama list` 和 HTTP API
- 理解 Ollama 的 HTTP 服务架构（端口 11434）

🗣 **讲课话术**

> Cell 3 提到：通常安装后 Ollama 会**自动启动服务**。也就是说你装完之后，它就已经在后台跑了。
>
> 但万一没有自动启动呢？手动运行：
>
> ```
> ollama serve
> ```
>
> 注意：如果服务已经在运行，再执行 `ollama serve` 会报端口占用的错误，这是**正常的**，说明服务已经在跑了。
>
> **怎么确认服务正常？** Cell 4 给了两种方法：
>
> **方法一**：终端里输入 `ollama list`，如果能看到你刚才拉取的 `qwen3:4b` 模型，说明服务正常。
>
> **方法二**：打开浏览器，访问 `http://localhost:11434/api/tags`。如果返回一段 JSON，里面包含 `qwen3:4b` 的信息，说明服务正常。
>
> 大家看 Cell 4 的截图（`assets/status.png`），这就是正常状态的样子。
>
> **类比**：Ollama 的服务其实就是一个本地的 HTTP 服务器，监听在 11434 端口。后面我们的 Python 代码会通过 HTTP 请求调用它——跟调用 OpenAI API 的原理一模一样，只不过请求发到 `localhost` 而不是 `api.openai.com`。
>
> **互动**：大家在浏览器里访问一下 `http://localhost:11434/api/tags`，看看能不能看到 JSON 返回？——看到 `qwen3:4b` 了吧？恭喜，你的本地大模型服务已经就绪了！

👀 **输出要点**
- Cell 3 是 Markdown，提示 `ollama serve` 命令
- Cell 4 是 Markdown + 截图 `assets/status.png`，展示 `ollama list` 和 HTTP API 的输出
- `ollama list` 应显示模型名称、大小、修改时间
- `http://localhost:11434/api/tags` 应返回 JSON，包含 models 数组

❓ **预判问题**

Q: `ollama serve` 报「端口已被占用」怎么办？
A: 说明服务已经在运行了！这是好事，不用管它。直接用 `ollama list` 验证即可。

Q: 浏览器访问 `localhost:11434` 没有响应？
A: 几种可能：1）服务没启动——运行 `ollama serve`；2）防火墙拦截了——暂时关闭防火墙试试；3）端口被其他程序占用——用 `netstat -ano | findstr 11434`（Windows）或 `lsof -i :11434`（Mac/Linux）查看。

Q: 11434 这个端口能改吗？
A: 可以。设置环境变量 `OLLAMA_HOST=0.0.0.0:自定义端口` 即可。但课程中我们统一用默认端口，避免配置不一致。

Q: 服务会一直在后台运行吗？消耗资源吗？
A: 服务本身很轻量，空闲时几乎不消耗 CPU/内存。只有你发请求让它推理时才会占用资源。模型在首次调用时加载到内存，空闲几分钟后会自动卸载。

➡️ **转场**

> 安装完成、模型就位、服务正常——三大步全部搞定。最后一步，看看在 Python notebook 里怎么调用它。

## 段落 4：在 notebook 中使用 + 总结 + Q&A（Cell 5）

---

📍 **展示 Cell 5**

⏱ **时间分配：3 分钟**

🎯 **本段目标**
- 展示后续 App 系列中调用 Ollama 的标准代码模式
- 总结本节完成的所有步骤
- 回答学生剩余问题

🗣 **讲课话术**

> 最后看一下 Cell 5——后续 App notebook 里调用 Ollama 的代码就这一行：
>
> ```python
> llm = get_llm_backend("ollama", model="qwen3:4b")
> ```
>
> `get_llm_backend` 是我们课程框架里封装好的函数。第一个参数指定后端类型（`"ollama"` 或 `"openai"`），第二个参数指定模型名。它内部会自动连接 `localhost:11434`，发 HTTP 请求。
>
> 这个设计的好处是：如果你以后想换成 OpenAI 的 GPT-4，只需要改一行：
> ```python
> llm = get_llm_backend("openai", model="gpt-4")
> ```
> 其他代码完全不用动。这就是**后端抽象**的价值。
>
> **好，来总结一下今天做了什么——四步走完：**
>
> 1. **安装 Ollama** —— 下载安装包，`ollama --version` 验证
> 2. **拉取模型** —— `ollama pull qwen3:4b`，下载约 2.5GB
> 3. **确认服务** —— `ollama list` 或访问 `http://localhost:11434/api/tags`
> 4. **代码调用** —— `get_llm_backend("ollama", model="qwen3:4b")`
>
> 从现在开始，你的电脑就是一台「本地 AI 服务器」了。后面的 App 系列——RAG、Agent、Function Calling——都会基于这个后端来跑。
>
> **互动**：有没有同学还有问题的？关于安装、下载、服务都可以问。——没有的话，我们就正式进入应用开发的世界了！

👀 **输出要点**
- Cell 5 展示调用代码：`llm = get_llm_backend("ollama", model="qwen3:4b")`
- 核心理解：Ollama 作为 HTTP 服务被 Python 代码调用，与调 OpenAI API 架构相同

❓ **预判问题**

Q: `get_llm_backend` 函数在哪里定义的？
A: 在课程的公共工具模块里。后面打开第一个 App notebook 时会看到 import 语句。不用自己写这个函数。

Q: 除了 qwen3:4b，后面的课程会用到其他模型吗？
A: 课程统一使用 qwen3:4b，保证大家环境一致。课后你可以自由实验其他模型，代码只需改 model 参数。

Q: 如果我课后想关掉 Ollama 服务怎么办？
A: Windows：任务栏右下角找到 Ollama 图标，右键退出。Mac：菜单栏点 Ollama 图标，选 Quit。Linux：`systemctl stop ollama` 或直接 kill 进程。

Q: Ollama 支持同时处理多个请求吗？
A: 支持。默认会排队处理。如果需要更高并发，可以配置 `OLLAMA_NUM_PARALLEL` 环境变量。但课程场景下单请求就够了。

➡️ **转场（连接后续课程）**

> 环境全部就绪！下一节我们正式进入第一个应用——你会看到这个 Ollama 后端是怎么被实际的 RAG/Agent 代码调用的。准备好了吗？我们开始！

## 附录 A：时间速查表

| 时间 | 段落 | 关键动作 |
|:---|:---|:---|
| 00:00 | 开场 | 展示 Cell 0，介绍 Ollama 定位 |
| 05:00 | 安装 | 展示 Cell 1，学生实操安装，`ollama --version` 验证 |
| 12:00 | 拉取模型 | 展示 Cell 2 + Cell 6，运行 `ollama pull qwen3:4b` |
| 18:00 | 启动与检查 | 展示 Cell 3 + Cell 4，`ollama list` + 浏览器访问 API |
| 22:00 | 调用示例 + 总结 | 展示 Cell 5，`get_llm_backend()` 用法，Q&A |

## 附录 B：关键数据速查

| 数据项 | 值 | 来源 Cell |
|:---|:---|:---|
| Ollama 官网 | https://ollama.ai/download | Cell 1 |
| 验证安装命令 | `ollama --version` | Cell 1 |
| 统一使用模型 | `qwen3:4b` | Cell 2 |
| 模型参数量 | 约 40 亿（4B） | Cell 2 |
| 模型下载大小 | 约 2.5GB | Cell 6 |
| 运行时内存占用 | 约 3-4GB | — |
| 拉取命令 | `ollama pull qwen3:4b` | Cell 2 |
| 手动启动服务 | `ollama serve` | Cell 3 |
| 服务默认端口 | 11434 | Cell 4 |
| 检查服务命令 | `ollama list` | Cell 4 |
| 检查服务 URL | `http://localhost:11434/api/tags` | Cell 4 |
| Python 调用代码 | `get_llm_backend("ollama", model="qwen3:4b")` | Cell 5 |
| 模型存储路径（Windows） | `C:\Users\用户名\.ollama\models` | — |
| 模型存储路径（Mac/Linux） | `~/.ollama/models` | — |

## 附录 C：应急预案

### 1. 安装问题

| 问题 | 解决方案 |
|:---|:---|
| Windows 下载安装包失败 | 使用国内镜像或让学生课后在网络好的环境安装；课上先看讲师演示 |
| Mac 提示「无法验证开发者」 | 系统偏好设置 -> 安全性与隐私 -> 通用 -> 仍要打开 |
| Linux `curl` 安装脚本失败 | 检查网络代理设置；或手动下载二进制文件 |
| `ollama --version` 提示命令未找到 | 重启终端；检查 PATH 环境变量；Windows 检查 `%LOCALAPPDATA%\Programs\Ollama` |
| 磁盘空间不足 | 清理磁盘，至少预留 5GB（安装包 + 模型文件） |

### 2. 模型下载问题

| 问题 | 解决方案 |
|:---|:---|
| `ollama pull` 速度极慢（<100KB/s） | 设置 HTTP 代理：`HTTPS_PROXY=http://proxy:port ollama pull qwen3:4b` |
| 下载中断 | 重新执行 `ollama pull qwen3:4b`，会从断点续传 |
| 下载完成但模型损坏 | `ollama rm qwen3:4b` 删除后重新 pull |
| 网络完全不通 | 讲师用 U 盘拷贝 `~/.ollama/models` 目录给学生（提前准备） |

### 3. 服务启动问题

| 问题 | 解决方案 |
|:---|:---|
| `ollama serve` 报端口占用 | 正常现象，说明服务已在运行，直接用 `ollama list` 验证 |
| `ollama list` 无输出或报错 | 确认 Ollama 进程在运行：Windows 用任务管理器查看，Mac/Linux 用 `ps aux \| grep ollama` |
| `localhost:11434` 无法访问 | 检查防火墙设置；尝试 `127.0.0.1:11434`；确认服务进程存在 |
| 推理时报 OOM（内存不足） | 关闭其他占内存的程序；或换更小的模型 `qwen3:1.7b` |

### 4. 教学进度问题

| 问题 | 解决方案 |
|:---|:---|
| 大量学生安装卡住（还剩 10 min） | 让已成功的学生帮助旁边的同学；讲师继续演示后续步骤，学生课后自行补装 |
| 模型下载太慢无法等待 | 讲师用自己电脑演示完整流程；学生课后下载，下节课前验证 |
| 学生提前全部完成 | 引导尝试 `ollama run qwen3:4b` 在终端直接对话体验模型能力 |
| 个别学生环境始终无法解决 | 提供备选方案：使用 OpenAI API 作为后端（需 API key），或与同桌共享一台电脑 |

### 5. 备用素材

- 提前录制一段完整安装 + 拉取 + 验证的屏幕录像（约 3 分钟），网络问题时播放
- 准备各平台（Windows/Mac/Linux）的安装截图，网络慢时用截图讲解
- 准备一份纯文字版的安装步骤清单，发给学生课后参考